## Load the required packages

In [1]:
import anndata
import pandas as pd
import numpy as np
import scanpy as sc
import torch
import torch.nn.functional as F
from pathlib import Path
import os

## Set path

In [2]:
working_dir = Path(os.getcwd())
base_dir = f"{working_dir.parents[0]}/outputs/batch variation/208_M2-208_M1"

## Load base results

In [3]:
base_methods = ["Babel", "cTPnet", "Dengkw", "Liger", "moETM", "sciPENN", "scMM", "scMOG", "scMoGNN", "scVAEIT", "Seurat", "TotalVI"]
base_results = {}
for name in base_methods:
    if name == "scMOG":
        results = anndata.read(f"{base_dir}/{name}/RNA_protein_adata.h5ad")
        results = pd.DataFrame(results.X.toarray(), index=results.obs.index, columns=results.var.index)
        base_results[name] = results
    
    elif name == "Liger":
        results = pd.read_csv(f"{base_dir}/{name}/test_predicted_protein_expression ({name}).csv", index_col=0).T
        base_results[name] = results
        
    elif name == "Seurat":
        results = pd.read_table(f"{base_dir}/{name}/test_predicted_protein_expression ({name}).txt", index_col=0, sep=" ").T
        base_results[name] = results
        
    else:
        results = pd.read_csv(f"{base_dir}/{name}/test_predicted_protein_expression ({name}).csv", index_col=0)
        base_results[name] = results

## Recover the protein names in cTPnet, Liger, and Seurat

In [4]:
cTPnet_protein_names_convert = pd.read_csv(f"{base_dir}/cTPnet/cTPnet_protein_names_convert.csv", index_col=0)
cTPnet_protein_names_convert.set_index("convert_name", inplace=True)
Liger_protein_names_convert = pd.read_csv(f"{base_dir}/Liger/Liger_protein_names_convert.csv", index_col=0)
Liger_protein_names_convert.set_index("convert_name", inplace=True)
Seurat_protein_names_convert = pd.read_csv(f"{base_dir}/Seurat/Seurat_protein_names_convert.csv", index_col=0)
Seurat_protein_names_convert.set_index("convert_name", inplace=True)

base_results["cTPnet"].columns = cTPnet_protein_names_convert.loc[base_results["cTPnet"].columns, "original_name"].values
base_results["Liger"].columns = Liger_protein_names_convert.loc[base_results["Liger"].columns, "original_name"].values
base_results["Seurat"].columns = Seurat_protein_names_convert.loc[base_results["Seurat"].columns, "original_name"].values

## Align rows and columns of all base results to the first one as reference

In [5]:
base_df = base_results[base_methods[0]]
base_index = base_df.index
base_columns = base_df.columns
for key, df in base_results.items():
    base_results[key] = df.reindex(index=base_index, columns=base_columns)

## Verify that rows and columns have the same order across base results

In [6]:
for key, df in base_results.items():
    same_index = (df.index.equals(base_index))
    same_columns = (df.columns.equals(base_columns))
    print(f"{key}: index aligned? {same_index}, columns aligned? {same_columns}")

Babel: index aligned? True, columns aligned? True
cTPnet: index aligned? True, columns aligned? True
Dengkw: index aligned? True, columns aligned? True
Liger: index aligned? True, columns aligned? True
moETM: index aligned? True, columns aligned? True
sciPENN: index aligned? True, columns aligned? True
scMM: index aligned? True, columns aligned? True
scMOG: index aligned? True, columns aligned? True
scMoGNN: index aligned? True, columns aligned? True
scVAEIT: index aligned? True, columns aligned? True
Seurat: index aligned? True, columns aligned? True
TotalVI: index aligned? True, columns aligned? True


## Convert log-scale base results back to the original scale by exponentiation

In [7]:
need_exp = ["Babel", "cTPnet", "Dengkw", "sciPENN", "scMOG", "scMoGNN", "scVAEIT", "Seurat"]
for key, df in base_results.items():
    if key in need_exp:
        base_results[key] = np.exp(df)
    
    else:
        base_results[key] = df.copy()

## Normalize base results

In [8]:
def log_normalize(data, target_sum=10000):
    adata = anndata.AnnData(X=data)
    sc.pp.normalize_total(adata, target_sum=target_sum)
    sc.pp.log1p(adata)
    return pd.DataFrame(adata.X, index=adata.obs.index, columns=adata.var.index)

def Z_score(data):
    adata = anndata.AnnData(X=data)
    sc.pp.scale(adata)
    return pd.DataFrame(adata.X, index=adata.obs.index, columns=adata.var.index)

base_results_ln = {k: log_normalize(v.copy()) for k, v in base_results.items()}
base_results_zs = {k: log_normalize(v.copy()) for k, v in base_results_ln.items()}

## Define a function to compute cosine distance

In [9]:
def cosine_distance_sum(H1: torch.Tensor, H2: torch.Tensor) -> torch.Tensor:
    assert H1.shape == H2.shape, "H1 and H2 must have the same shape"
    N, P = H1.shape
#
    row_norm_H1 = H1.norm(dim=1)
    row_norm_H2 = H2.norm(dim=1)
    row_valid = (row_norm_H1 > 0) & (row_norm_H2 > 0)
#
    A = F.normalize(H1[row_valid], dim=1)
    B = F.normalize(H2[row_valid], dim=1)
    row_cos_sim = (A * B).sum(dim=1)
    row_dist = torch.full((N,), 2.0, device=H1.device)
    row_dist[row_valid] = 1 - row_cos_sim
    row_loss = row_dist.mean()
#
    H1_T = H1.T
    H2_T = H2.T
    col_norm_H1 = H1_T.norm(dim=1)
    col_norm_H2 = H2_T.norm(dim=1)
    col_valid = (col_norm_H1 > 0) & (col_norm_H2 > 0)
#
    A_T = F.normalize(H1_T[col_valid], dim=1)
    B_T = F.normalize(H2_T[col_valid], dim=1)
    col_cos_sim = (A_T * B_T).sum(dim=1)
    col_dist = torch.full((P,), 2.0, device=H1.device)
    col_dist[col_valid] = 1 - col_cos_sim
    col_loss = col_dist.mean()
#
    return row_loss + col_loss

## Calculate the naive average of the base results

In [10]:
base_results_zs = list(base_results_zs.values())
M = len(base_results_zs)
mean_res = sum(base_results_zs) / len(base_results_zs)
mean_res

,ADT_4-1BBLigand(CD137L)_A0890,ADT_CD102_A0104,ADT_CD103_A0201,ADT_CD105_A0812,ADT_CD106_A0226,ADT_CD107a(LAMP-1)_A0905,ADT_CD115(CSF-1R)_A0105,ADT_CD117(c-Kit)_A0012,ADT_CD11a_A0595,ADT_CD11b_A0014,...,ADT_TCRVr3_A0210,ADT_TCRbchain_A0120,ADT_TCRr-d_A0121,ADT_TER-119-ErythroidCells_A0122,ADT_TIGIT(Vstm3)_A0848,ADT_TLR4(CD284)-MD2Complex_A0875,ADT_Tim-4_A0567,ADT_XCR1_A0568,ADT_anti-P2RY12_A0415,ADT_integrinb7_A0214
index,,,,,,,,,,,,,,,,,,,,,
AAACCCAAGCGATCGA-1,3.762014,4.179905,4.235006,3.833395,3.847665,3.463089,3.758950,3.974681,4.451051,3.463900,...,4.026074,4.254517,3.862129,3.521792,4.066590,3.774194,3.920286,3.837776,3.910805,4.410877
AAACCCACACCTCTAC-1,3.793056,4.302042,3.946241,3.790181,3.909536,3.802155,3.806943,3.831318,4.361076,3.853334,...,3.978614,3.804478,3.827622,3.252525,4.041994,3.778148,4.012038,3.798774,3.890942,4.131676
AAACCCACACGTCGTG-1,3.789724,4.394031,3.966532,3.498961,3.885261,3.748854,3.785270,3.822779,4.262487,3.865209,...,4.014831,3.503383,3.861780,3.241684,4.022119,3.819564,3.958819,3.787467,3.902275,4.164600
AAACCCACATGTCTAG-1,3.799661,4.363760,3.957075,3.786614,3.888182,3.807688,3.785386,3.760519,4.290310,3.847821,...,4.005929,3.779281,3.858604,3.513225,4.054255,3.750814,3.993867,3.781849,3.916672,4.164496
AAACCCAGTCAAAGTA-1,3.456553,4.305390,3.942046,3.760663,3.876785,3.769105,3.817566,3.882949,4.337999,3.508797,...,4.010127,3.869772,3.525175,3.207823,4.052586,3.773986,3.966206,3.836206,3.919435,4.239667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGCATTCCTCG-1,3.779138,4.319893,3.926904,3.827325,3.870665,3.470368,3.783847,3.847224,4.318328,3.936066,...,3.996255,3.812528,3.862926,3.217557,4.034974,3.800616,3.969103,3.484411,3.916634,4.237754
TTTGTTGGTGCTCTTC-1,3.765122,4.310352,3.946872,3.777145,3.883072,3.815306,3.798374,3.784503,4.288279,3.868408,...,3.991827,3.826446,3.834443,3.454972,4.048337,3.804723,3.972051,3.807993,3.892137,4.210668
TTTGTTGTCAGCCTTC-1,3.816451,4.181170,3.944892,3.784502,3.874308,3.779820,3.808163,3.937730,4.447227,3.856363,...,4.035530,4.380803,3.522913,3.503627,4.076241,3.833734,3.984084,3.846434,3.902951,4.125801


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
H0 = torch.tensor(mean_res.values, dtype=torch.float32, device=device)
H_list = [torch.tensor(h.values, dtype=torch.float32, device=device) for h in base_results_zs]
alpha = torch.zeros(M, requires_grad=True, device=device)
optimizer = torch.optim.Adam([alpha], lr=0.1)

## Empirically set the values of lambda

In [12]:
cos_dist_init = [cosine_distance_sum(H0, Hm).item() for Hm in H_list]
lamda = np.median(cos_dist_init)
cos_dist_init, lamda

([0.0010900360066443682,
  0.000797063228674233,
  0.0010905256494879723,
  0.0015214059967547655,
  0.0018858581315726042,
  0.0025300614070147276,
  0.08953037112951279,
  0.0008601216832175851,
  0.0011674228589981794,
  0.003467781934887171,
  0.0011312206042930484,
  0.0008821339579299092],
 0.001149321731645614)

## Start optimization

In [13]:
loss_list = []
cos_dist_list = []
for step in range(500):
    optimizer.zero_grad()
#
    weights = F.softmax(alpha, dim=0)  # (M,)
#
    H_ens = sum(w * H for w, H in zip(weights, H_list))  # (N, D)
#
    cos_dist = 0.0
    for w, Hm in zip(weights, H_list):
        dist = cosine_distance_sum(H_ens, Hm)
        cos_dist += w * dist
#   
    loss = cos_dist + lamda * (weights * torch.log(weights + 1e-8)).sum()
#
    loss.backward()
    optimizer.step()
    loss_list.append(loss.item())
    cos_dist_list.append(cos_dist.item())
#
    print(f"Step {step}: Loss = {loss.item():.4f}, Weights = {weights.detach().cpu().numpy()}")

Step 0: Loss = 0.0060, Weights = [0.08333334 0.08333334 0.08333334 0.08333334 0.08333334 0.08333334
 0.08333334 0.08333334 0.08333334 0.08333334 0.08333334 0.08333334]
Step 1: Loss = 0.0047, Weights = [0.08461145 0.08461145 0.08461145 0.08461144 0.08461143 0.08461141
 0.06927411 0.08461145 0.08461145 0.08461139 0.08461145 0.08461145]
Step 2: Loss = 0.0037, Weights = [0.08569039 0.08568902 0.08569048 0.08567866 0.0856713  0.08567554
 0.0575055  0.08568843 0.08569004 0.08564484 0.08569023 0.08568561]
Step 3: Loss = 0.0028, Weights = [0.08660594 0.08660088 0.08660625 0.08655986 0.08653041 0.08654623
 0.04773413 0.08659858 0.08660452 0.08642032 0.08660528 0.08658762]
Step 4: Loss = 0.0020, Weights = [0.08739516 0.08738321 0.08739591 0.08727948 0.08720413 0.08724205
 0.03967507 0.08737739 0.0873915  0.08691242 0.08739343 0.08735023]
Step 5: Loss = 0.0014, Weights = [0.08809482 0.08807193 0.08809633 0.08786052 0.08770534 0.08777903
 0.03306198 0.08806009 0.08808724 0.08708584 0.08809123 0.08

## Save the loss during optimization

In [14]:
loss_list = np.array(loss_list)
cos_dist_list = np.array(cos_dist_list)
np.savetxt(f"{base_dir}/AdaPro_total_loss.csv", loss_list, delimiter='\t')
np.savetxt(f"{base_dir}/AdaPro_cosine_loss.csv", cos_dist_list, delimiter='\t')

## Save the consensus weights and predictions

In [15]:
AdaPro_weights = F.softmax(alpha, dim=0).detach().cpu().numpy()
AdaPro_weights_df = pd.DataFrame(AdaPro_weights, index=base_methods, columns=["AdaPro_weight"])
AdaPro_weights_df.to_csv(f"{base_dir}/AdaPro_weights.csv")
AdaPro_weights_df

,AdaPro_weight
Babel,0.143960
cTPnet,0.114862
Dengkw,0.143905
Liger,0.042306
moETM,0.024660
sciPENN,0.055338
scMM,0.000076
scMOG,0.109961
scMoGNN,0.139419
scVAEIT,0.007298


In [16]:
AdaPro_prediction = pd.DataFrame(sum(w * H for w, H in zip(AdaPro_weights, H_list)).detach().cpu().numpy(), index=mean_res.index, columns=mean_res.columns)
AdaPro_prediction.to_csv(f"{base_dir}/test_predicted_protein_expression (AdaPro).csv")
AdaPro_prediction

,ADT_4-1BBLigand(CD137L)_A0890,ADT_CD102_A0104,ADT_CD103_A0201,ADT_CD105_A0812,ADT_CD106_A0226,ADT_CD107a(LAMP-1)_A0905,ADT_CD115(CSF-1R)_A0105,ADT_CD117(c-Kit)_A0012,ADT_CD11a_A0595,ADT_CD11b_A0014,...,ADT_TCRVr3_A0210,ADT_TCRbchain_A0120,ADT_TCRr-d_A0121,ADT_TER-119-ErythroidCells_A0122,ADT_TIGIT(Vstm3)_A0848,ADT_TLR4(CD284)-MD2Complex_A0875,ADT_Tim-4_A0567,ADT_XCR1_A0568,ADT_anti-P2RY12_A0415,ADT_integrinb7_A0214
index,,,,,,,,,,,,,,,,,,,,,
AAACCCAAGCGATCGA-1,3.817907,4.143697,4.196487,3.836273,3.841838,3.788819,3.808467,3.934795,4.403585,3.816144,...,3.996256,4.222044,3.862307,3.620531,4.021715,3.832752,3.935245,3.822791,3.906328,4.345287
AAACCCACACCTCTAC-1,3.791790,4.256878,3.923082,3.804033,3.904397,3.809482,3.797613,3.822394,4.350668,3.825217,...,3.959751,3.811607,3.825924,3.635430,4.005447,3.789449,3.998415,3.828075,3.867786,4.095747
AAACCCACACGTCGTG-1,3.810685,4.368353,3.946653,3.831862,3.864308,3.796092,3.799678,3.834121,4.210630,3.856643,...,3.981858,3.839611,3.861253,3.632844,4.008352,3.820262,3.937443,3.830501,3.895743,4.141787
AAACCCACATGTCTAG-1,3.792098,4.328015,3.929732,3.810225,3.864941,3.799546,3.785662,3.829530,4.271659,3.833857,...,3.970244,3.810950,3.854955,3.632943,4.013797,3.814045,3.955496,3.820640,3.896291,4.134576
AAACCCAGTCAAAGTA-1,3.799362,4.256075,3.934438,3.810160,3.867351,3.813530,3.800527,3.846454,4.297073,3.834540,...,3.979129,3.834408,3.849883,3.615787,4.011728,3.800612,3.947466,3.830513,3.899673,4.184806
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGCATTCCTCG-1,3.786403,4.263252,3.934550,3.832594,3.865317,3.808351,3.810758,3.846072,4.270144,3.914462,...,3.977782,3.839362,3.858001,3.623211,4.016964,3.812399,3.958728,3.820159,3.899483,4.177395
TTTGTTGGTGCTCTTC-1,3.799323,4.266675,3.934314,3.825869,3.868092,3.819027,3.817139,3.854803,4.252442,3.860951,...,3.982053,3.841914,3.856089,3.607587,4.023036,3.810563,3.959390,3.823554,3.900990,4.181790
TTTGTTGTCAGCCTTC-1,3.814278,4.148173,3.928086,3.818609,3.844681,3.802485,3.826708,3.912287,4.413199,3.864571,...,4.000381,4.322999,3.852013,3.633841,4.024358,3.835100,3.948349,3.833802,3.890440,4.101065
